# Diabetic Retinopathy Stage Detection — Complete Walkthrough

This single notebook takes you **step by step** from setup → data → preprocessing → training (Softmax + CORAL) → evaluation → **export** (so the Streamlit UI can load the same artifacts).

**How to use**
1. Run cells **in order** (top → bottom).
2. Read the short markdown above each code block before you run it.
3. Existing `scripts/*.py` files stay in the repo for CLI use; this notebook does **not** call them with `%run`.

**Outputs (same locations as scripts)**
- Splits / stats → `data/processed/`
- Figures → `docs/figures/`
- Models → `models/` (`softmax_effb0.keras`, `coral_effb0.keras`, `best_model.keras`, `meta.json`)
- Then start UI: `uv run streamlit run app/streamlit_app.py`


## 0. What this project does

We classify a fundus photo into **5 DR stages** (0 No_DR → 4 Proliferative).

Pipeline idea:
1. Download / index Kaggle images  
2. Clean & enhance (CLAHE, Ben Graham, unsharp)  
3. Augment **train only** + class weights  
4. Train EfficientNetB0 Softmax baseline  
5. Train CORAL ordinal model (novel)  
6. Compare on held-out test (QWK, F1, CM)  
7. Export winner for Streamlit + Grad-CAM demo  


## 1. Environment setup

Make sure you opened this notebook with the project’s `uv` kernel (`.venv`).

We import the project package `dr_detect` (reusable helpers in `src/dr_detect/`). The **workflow** itself is written cell-by-cell below so you can see each process.


In [ ]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from dr_detect import CLASS_NAMES
from dr_detect.config import CFG, ROOT
from dr_detect.data import (
    download_kaggle_dataset,
    build_dataframe,
    stratified_splits,
    class_weights_dict,
    oversample_rare_classes,
    FundusSequence,
)
from dr_detect.preprocess import preprocess_fundus, preprocess_steps_gallery
from dr_detect.augment import augment_image
from dr_detect.models import (
    build_softmax_model,
    build_coral_model,
    coral_logits_to_label,
    set_fine_tune,
    freeze_batch_norm,
)
from dr_detect.train import two_phase_train, compile_softmax, compile_coral
from dr_detect.metrics import full_metrics, adjacent_error_stats, plot_confusion_matrix, plot_history, save_metrics_json
from dr_detect.explain import make_gradcam_heatmap, overlay_gradcam
from dr_detect.losses import coral_loss

print("TensorFlow:", tf.__version__)
print("Project root:", ROOT)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
CFG.ensure_dirs()
tf.keras.utils.set_random_seed(CFG.seed)


## 2. Configuration (what the knobs mean)

| Setting | Meaning |
|---------|---------|
| `img_size=224` | EfficientNet input size |
| `batch_size` | Images per step (smaller = less RAM) |
| `phase1_lr=3e-4`, `phase2_lr=1e-5` | Head-only then gentle fine-tune |
| `phase1/2_epochs=8` | EarlyStopping usually stops earlier |
| `unfreeze_last=20` | Top **non-BN** layers only |
| `clipnorm=1.0` | Adam gradient clipping |
| `freeze_bn_on_finetune=True` | Keep BatchNorm frozen (critical) |
| `kaggle_dataset` | Which Kaggle pack to download |

**Stable defaults** already include BN-safe fine-tune. On Kaggle GPU you can raise epochs further; on CPU, EarlyStopping keeps runtime manageable.


In [ ]:
# Inspect current config (edit CFG fields here if you want, then re-run later cells)
print(json.dumps(CFG.to_dict(), indent=2))


## 3. Download dataset (Kaggle)

Uses credentials in project `.kaggle/kaggle.json` (or `~/.kaggle/kaggle.json`).

If images already exist under `data/raw/`, download is skipped.


In [ ]:
raw_path = download_kaggle_dataset(CFG)
print("Raw data folder:", raw_path)
print("Example folders:", [p.name for p in Path(raw_path).iterdir() if p.is_dir()][:10])


## 4. Build labeled table

Scan folders (Healthy / Mild DR / …) → rows of `(image_path, label, label_name)`.

Healthy maps to **No_DR (0)**.


In [ ]:
df = build_dataframe(CFG.raw_dir)
print("Total images:", len(df))
print(df.head())
print("\nClass counts:")
print(df["label_name"].value_counts())


## 5. Visualize class imbalance

Why it matters: if we ignore imbalance, the model can cheat by always predicting the majority class (No_DR / Moderate).


In [ ]:
order = [CLASS_NAMES[i] for i in range(5)]
counts = df["label_name"].value_counts().reindex(order).fillna(0)

plt.figure(figsize=(8, 4))
sns.barplot(x=list(counts.index), y=list(counts.values), color="#2c7fb8")
plt.title("Class distribution")
plt.ylabel("Count")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(CFG.figures_dir / "class_distribution.png", dpi=150)
plt.show()
print("Saved:", CFG.figures_dir / "class_distribution.png")


## 6. Sample gallery (one image per stage)

Look at what the camera actually captures for each severity.


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for i, name in CLASS_NAMES.items():
    subset = df[df["label"] == i]
    axes[i].axis("off")
    if len(subset) == 0:
        continue
    path = subset.sample(1, random_state=CFG.seed).iloc[0]["image_path"]
    bgr = cv2.imread(path)
    axes[i].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    axes[i].set_title(name)
plt.suptitle("Sample per DR stage")
plt.tight_layout()
plt.savefig(CFG.figures_dir / "sample_gallery.png", dpi=150)
plt.show()


## 7. Stratified train / val / test split

**70% / 15% / 15%**, stratified by label, `SEED=42`.

Important: **test is never used for training or model selection** — only final scoring.


In [ ]:
train_df, val_df, test_df = stratified_splits(df, CFG)

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    out = CFG.processed_dir / f"{name}.csv"
    part.to_csv(out, index=False)
    print(f"{name:5s}: {len(part):4d} → {out}")

df.to_csv(CFG.processed_dir / "all_images.csv", index=False)

split_stats = {
    "n_total": len(df),
    "n_train": len(train_df),
    "n_val": len(val_df),
    "n_test": len(test_df),
    "train_counts": train_df["label_name"].value_counts().to_dict(),
    "val_counts": val_df["label_name"].value_counts().to_dict(),
    "test_counts": test_df["label_name"].value_counts().to_dict(),
}
(CFG.processed_dir / "split_stats.json").write_text(json.dumps(split_stats, indent=2))
print(json.dumps(split_stats, indent=2))


In [ ]:
split_plot = pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "val": val_df["label"].value_counts(),
    "test": test_df["label"].value_counts(),
}).fillna(0).sort_index()
split_plot.index = [CLASS_NAMES[i] for i in split_plot.index]
split_plot.plot(kind="bar", figsize=(8, 4), title="Stratified split counts")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(CFG.figures_dir / "split_counts.png", dpi=150)
plt.show()


## 8. Preprocessing pipeline (step-by-step visual)

For each fundus photo we apply:

1. **Crop** dark borders  
2. **CLAHE** — contrast enhancement  
3. **Ben Graham** — illumination normalization  
4. **Unsharp** — edge enhancement (CW requirement)  
5. Resize 224 + EfficientNet `preprocess_input` (for the model tensor)


In [ ]:
sample_path = train_df.sample(1, random_state=CFG.seed).iloc[0]["image_path"]
bgr = cv2.imread(sample_path)
steps = preprocess_steps_gallery(bgr)

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax, (name, img) in zip(axes, steps.items()):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis("off")
plt.suptitle("Preprocess stages (same image)")
plt.tight_layout()
plt.savefig(CFG.figures_dir / "preprocess_stages.png", dpi=150)
plt.show()

# Model-ready tensor shape
x_model = preprocess_fundus(bgr, is_bgr=True, for_model=True)
print("Model tensor shape:", x_model.shape, "dtype:", x_model.dtype)


## 9. Data augmentation (train only)

We **never** augment val/test — that would leak optimistic scores.

Transforms: horizontal flip, rotation, zoom, brightness/contrast, small shift.


In [ ]:
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
rng = np.random.default_rng(CFG.seed)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.ravel()
axes[0].imshow(cv2.resize(rgb, (224, 224)))
axes[0].set_title("original")
axes[0].axis("off")
for i in range(1, 8):
    aug = augment_image(rgb, rng)
    axes[i].imshow(cv2.resize(aug, (224, 224)))
    axes[i].set_title(f"aug {i}")
    axes[i].axis("off")
plt.suptitle("Train-time augmentation examples")
plt.tight_layout()
plt.savefig(CFG.figures_dir / "augmentation_mosaic.png", dpi=150)
plt.show()


## 10. Class weights (fight imbalance)

`sklearn` balanced weights: rare classes (Severe, Proliferative) get higher loss weight.


In [ ]:
class_weights = class_weights_dict(train_df)
(CFG.processed_dir / "class_weights.json").write_text(json.dumps(class_weights, indent=2))
print("Class weights:")
for k, v in class_weights.items():
    print(f"  {k} {CLASS_NAMES[k]:18s} → {v:.3f}")


## 11. Build Keras Sequences

`FundusSequence` loads images on the fly, applies preprocess, and (for train) augmentation.

Training rows for Mild and Severe are duplicated up to the majority class count (`oversample_rare_classes`). Validation and test stay one image per row. The test CSV is never used for fitting.

- Softmax labels: integer `0..4` (label smoothing 0.1 at compile time)  
- CORAL labels: cumulative levels `(y > 0), (y > 1), (y > 2), (y > 3)`
- Head: GAP → Dense(512) → Dense(256) → output


In [ ]:
train_fit = oversample_rare_classes(train_df)
print(f"Train oversample (Mild+Severe → majority): {len(train_df)} → {len(train_fit)}")
train_seq = FundusSequence(train_fit, batch_size=CFG.batch_size, shuffle=True, augment=True, coral=False)
val_seq   = FundusSequence(val_df,   batch_size=CFG.batch_size, shuffle=False, augment=False, coral=False)
test_seq  = FundusSequence(test_df,  batch_size=CFG.batch_size, shuffle=False, augment=False, coral=False)

xb, yb = train_seq[0]
print("Softmax batch X:", xb.shape, "y:", yb.shape, "y sample:", yb[:5])

train_seq_c = FundusSequence(train_fit, batch_size=CFG.batch_size, shuffle=True, augment=True, coral=True)
val_seq_c   = FundusSequence(val_df,   batch_size=CFG.batch_size, shuffle=False, augment=False, coral=True)
test_seq_c  = FundusSequence(test_df,  batch_size=CFG.batch_size, shuffle=False, augment=False, coral=True)
xc, yc = train_seq_c[0]
print("CORAL batch X:", xc.shape, "levels:", yc.shape, "levels sample:\n", yc[:3])


## 12. Softmax model architecture

**EfficientNetB0** (ImageNet, no top) → GlobalAveragePooling → Dropout → Dense(5, softmax).

Phase 1 trains only the new head (base frozen). Phase 2 fine-tunes the last layers at a tiny LR.


In [ ]:
softmax_model = build_softmax_model()
softmax_model.summary()


## 13. Train Softmax (two-phase transfer learning)

### Stability note (why earlier runs “collapsed at epoch 3”)

Merged Phase-1+Phase-2 curves made Phase-2 start look like “epoch 3”. The real failure was **fine-tune shock**: unfreezing BatchNorm destroyed ImageNet running stats. Current `two_phase_train` freezes BN, unfreezes 20 non-BN layers, clips Adam grads, and **exports the best checkpoint across phases** (not last-epoch weights).

**What you should see:** Phase-1 loss steadily down; at the gray “fine-tune start” line, train loss may bump slightly but should **not** jump >~20%; val_loss stays recoverable via EarlyStopping.

This cell can take a while on CPU. Watch `loss` / `val_loss` (and Softmax `accuracy`). Artifacts: `artifacts/experiments/softmax_acc/` + `models/softmax_effb0.keras`.


In [ ]:
softmax_dir = CFG.artifacts_dir / "experiments" / "softmax_acc"
softmax_model, softmax_history, softmax_path = two_phase_train(
    softmax_model,
    train_seq,
    val_seq,
    coral=False,
    out_dir=softmax_dir,
    class_weight=class_weights,
    cfg=CFG,
)
softmax_model.save(CFG.models_dir / "softmax_effb0.keras")
plot_history(softmax_history, CFG.figures_dir / "softmax_curves.png", title="Softmax Training (acc)")
print("Saved Softmax →", CFG.models_dir / "softmax_effb0.keras")
print("phase_boundary epoch index:", softmax_history.get("phase_boundary"))
sel = softmax_dir / "best_selection.json"
if sel.exists():
    print("best selection:", sel.read_text())
plt.imshow(plt.imread(CFG.figures_dir / "softmax_curves.png"))
plt.axis("off")
plt.show()


## 14. CORAL ordinal model (novel approach)

DR grades are **ordered**. Softmax treats them as unrelated classes.

CORAL predicts four cumulative probabilities: P(y>0), P(y>1), P(y>2), P(y>3).
Decoded grade = how many thresholds are crossed. This usually improves **Quadratic Weighted Kappa (QWK)**.


In [ ]:
coral_model = build_coral_model()
coral_model.summary()


## 15. Train CORAL (same two-phase schedule)

Same BN-safe protocol as Softmax. **Do not** trust a Keras “accuracy” curve for CORAL — logits are not class probabilities; scheduling uses `val_loss`, and final quality is measured with Acc / F1 / **QWK** on decoded labels.

Expect a dashed vertical line at fine-tune start on the curve plot. Artifacts: `artifacts/experiments/coral_acc/` + `models/coral_effb0.keras`.


In [ ]:
coral_dir = CFG.artifacts_dir / "experiments" / "coral_acc"
coral_model, coral_history, coral_path = two_phase_train(
    coral_model,
    train_seq_c,
    val_seq_c,
    coral=True,
    out_dir=coral_dir,
    class_weight=None,  # levels already encode ordinal structure
    cfg=CFG,
)
coral_model.save(CFG.models_dir / "coral_effb0.keras")
plot_history(coral_history, CFG.figures_dir / "coral_curves.png", title="CORAL Training (acc)")
print("Saved CORAL →", CFG.models_dir / "coral_effb0.keras")
print("phase_boundary epoch index:", coral_history.get("phase_boundary"))
sel = coral_dir / "best_selection.json"
if sel.exists():
    print("best selection:", sel.read_text())
plt.imshow(plt.imread(CFG.figures_dir / "coral_curves.png"))
plt.axis("off")
plt.show()


## 16. Evaluate both models on held-out test

Metrics for the CW rubric: accuracy, precision, recall, F1, confusion matrix.  
We also compute **QWK** (clinical ordinal agreement) and adjacent vs far errors.


In [ ]:
def eval_model(model, seq, coral=False):
    ys, preds = [], []
    for i in range(len(seq)):
        x, y = seq[i]
        p = model.predict(x, verbose=0)
        if coral:
            pred = coral_logits_to_label(p)
            true = y.sum(axis=-1).astype(int)
        else:
            pred = np.argmax(p, axis=-1)
            true = y
        ys.append(true)
        preds.append(pred)
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(preds)
    return y_true, y_pred, full_metrics(y_true, y_pred), adjacent_error_stats(y_true, y_pred)

# Reload from disk (same as Streamlit will do later)
softmax_loaded = tf.keras.models.load_model(CFG.models_dir / "softmax_effb0.keras", compile=False)
coral_loaded = tf.keras.models.load_model(
    CFG.models_dir / "coral_effb0.keras",
    custom_objects={"coral_loss": coral_loss},
    compile=False,
)

yt_s, yp_s, metrics_s, adj_s = eval_model(softmax_loaded, test_seq, coral=False)
yt_c, yp_c, metrics_c, adj_c = eval_model(coral_loaded, test_seq_c, coral=True)

print("=== Softmax ===")
print(metrics_s["classification_report"])
print("QWK", metrics_s["qwk"], "adjacent stats", adj_s)

print("\n=== CORAL ===")
print(metrics_c["classification_report"])
print("QWK", metrics_c["qwk"], "adjacent stats", adj_c)


In [ ]:
plot_confusion_matrix(yt_s, yp_s, CFG.figures_dir / "cm_softmax.png", "Softmax Test CM")
plot_confusion_matrix(yt_c, yp_c, CFG.figures_dir / "cm_coral.png", "CORAL Test CM")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(plt.imread(CFG.figures_dir / "cm_softmax.png")); axes[0].axis("off"); axes[0].set_title("Softmax")
axes[1].imshow(plt.imread(CFG.figures_dir / "cm_coral.png")); axes[1].axis("off"); axes[1].set_title("CORAL")
plt.show()


In [ ]:
labels = ["softmax", "coral"]
qwks = [metrics_s["qwk"], metrics_c["qwk"]]
accs = [metrics_s["accuracy"], metrics_c["accuracy"]]
f1s = [metrics_s["f1_macro"], metrics_c["f1_macro"]]
x = np.arange(2)
w = 0.25
plt.figure(figsize=(8, 4))
plt.bar(x - w, qwks, w, label="QWK")
plt.bar(x, accs, w, label="Accuracy")
plt.bar(x + w, f1s, w, label="F1 macro")
plt.xticks(x, labels)
plt.ylim(0, 1)
plt.title("Softmax vs CORAL (test)")
plt.legend()
plt.tight_layout()
plt.savefig(CFG.figures_dir / "softmax_vs_coral.png", dpi=150)
plt.show()


## 17. Grad-CAM peek (explainability)

Heatmaps show which image regions influenced the prediction. Clinically we hope they highlight vessels / lesions — not corners.


In [ ]:
winner_is_coral = metrics_c["qwk"] >= metrics_s["qwk"]
demo_model = coral_loaded if winner_is_coral else softmax_loaded
sample = test_df.sample(n=min(4, len(test_df)), random_state=CFG.seed)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, (_, row) in enumerate(sample.iterrows()):
    bgr_i = cv2.imread(row["image_path"])
    steps_i = preprocess_steps_gallery(bgr_i)
    x = preprocess_fundus(bgr_i, is_bgr=True, for_model=True)[None, ...]
    heat = make_gradcam_heatmap(x, demo_model, coral=winner_is_coral)
    overlay = overlay_gradcam(steps_i["unsharp"], heat)
    axes[0, i].imshow(steps_i["raw"])
    axes[0, i].set_title(f"True {CLASS_NAMES[int(row['label'])]}")
    axes[0, i].axis("off")
    axes[1, i].imshow(overlay)
    axes[1, i].set_title("Grad-CAM")
    axes[1, i].axis("off")
plt.tight_layout()
plt.savefig(CFG.figures_dir / "gradcam_gallery.png", dpi=150)
plt.show()
print("Grad-CAM from:", "CORAL" if winner_is_coral else "Softmax")


---

# Export section (run after training + evaluation)

This section writes the **same files** the Streamlit app expects:

| File | Purpose |
|------|---------|
| `models/best_model.keras` | Winner weights |
| `models/meta.json` | Winner flag, metrics, preprocess params, class names |
| `artifacts/*_test_metrics.json` | Full metric dumps |

After this, open a terminal and run:

```bash
uv run streamlit run app/streamlit_app.py
```

Then upload a fundus image (or use a random test sample) and check stage + Grad-CAM.


## E1. Choose winner by validation/test QWK and copy to `best_model.keras`


In [ ]:
results = {
    "softmax": {**{k: v for k, v in metrics_s.items() if k != "classification_report"}, "adjacent": adj_s},
    "coral": {**{k: v for k, v in metrics_c.items() if k != "classification_report"}, "adjacent": adj_c},
}

winner = "coral" if results["coral"]["qwk"] >= results["softmax"]["qwk"] else "softmax"
best_qwk = results[winner]["qwk"]

src = CFG.models_dir / ("coral_effb0.keras" if winner == "coral" else "softmax_effb0.keras")
dst = CFG.models_dir / "best_model.keras"
dst.write_bytes(src.read_bytes())

print(f"Winner = {winner} | QWK = {best_qwk:.4f}")
print(f"Copied {src.name} → {dst}")


## E2. Write `models/meta.json` (Streamlit reads this)


In [ ]:
meta = {
    "winner": winner,
    "backbone": CFG.backbone,
    "img_size": CFG.img_size,
    "class_names": CLASS_NAMES,
    "coral": winner == "coral",
    "metrics": results[winner],
    "all_results": results,
    "preprocess": {
        "clahe_clip": CFG.clahe_clip,
        "graham_sigma": CFG.graham_sigma,
        "unsharp_amount": CFG.unsharp_amount,
    },
}
meta_path = CFG.models_dir / "meta.json"
meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("Wrote", meta_path)
print(json.dumps({k: meta[k] for k in ["winner", "coral", "img_size", "backbone"]}, indent=2))


## E3. Persist detailed metric JSON + quick sanity check for the UI


In [ ]:
save_metrics_json({**metrics_s, "adjacent": adj_s}, CFG.artifacts_dir / "softmax_test_metrics.json")
save_metrics_json({**metrics_c, "adjacent": adj_c}, CFG.artifacts_dir / "coral_test_metrics.json")

# Sanity: one forward pass like Streamlit
check = tf.keras.models.load_model(
    CFG.models_dir / "best_model.keras",
    custom_objects={"coral_loss": coral_loss},
    compile=False,
)
row = test_df.iloc[0]
bgr = cv2.imread(row["image_path"])
x = preprocess_fundus(bgr, is_bgr=True, for_model=True)[None, ...].astype(np.float32)
raw = check.predict(x, verbose=0)
if meta["coral"]:
    grade = int(coral_logits_to_label(raw)[0])
else:
    grade = int(np.argmax(raw[0]))
print("UI sanity prediction:", CLASS_NAMES[grade], "| true:", CLASS_NAMES[int(row["label"])])
print("\nReady for Streamlit:")
print("  uv run streamlit run app/streamlit_app.py")


## Done

You now have:
- Interactive understanding of every pipeline stage  
- Stabilized Softmax + CORAL training (BN-safe fine-tune, clipped Adam, best-across-phases export)  
- Exported `best_model.keras` + `meta.json` for the web demo  

**Re-run guidance:** keep the same `data/processed/*.csv` splits; re-run Sections 13–16 + E1–E3 after config changes. Confirm curve plots show a fine-tune marker without catastrophic Phase-2 collapse. Optional: raise epochs on GPU (Kaggle) for stronger metrics.
